In [ ]:
from jaxtyping import Float, Array, Key, Scalar
import jax
import jax.numpy as jnp
import jax.scipy as jsp
import jax.random as jr
from jax.numpy.fft import rfftfreq, fftfreq, irfft, rfft, fft
from flax import nnx
import optax
from einops import rearrange, einsum

import numpy as np
from tqdm import tqdm
from matplotlib import pyplot as plt
from corner import corner
from emcee import EnsembleSampler

from jax import config
config.update("jax_enable_x64",True)

In [ ]:
YEARSECONDS = 365 * 24 * 3600
WEEKSECONDS = 7 * 24 * 3600
YEAROMEGA = 2 * np.pi / YEARSECONDS
SUNEARTHRADIUS = 1.496e11
c = 3e8

ARMLENGHT = 2.5e9
ARMLENGHTNORMALIZED = ARMLENGHT / SUNEARTHRADIUS
ALPHATRIANGLE = 0
BETATRIANGLE = 0

NTIMESAMPS = int(WEEKSECONDS / 50)
TOBS = WEEKSECONDS
DT = TOBS / NTIMESAMPS
TIMES = jnp.linspace(0, TOBS, NTIMESAMPS, endpoint=False)
CHANNELS = 2


### LISA motion and geometry ###
def sat1_pos(t):
    alp = ALPHATRIANGLE
    bet = BETATRIANGLE
    omS = YEAROMEGA
    Lrs = ARMLENGHTNORMALIZED
    return jnp.stack(
        [
            jnp.cos(omS * t + alp)
            + Lrs
            / (4 * jnp.sqrt(3))
            * (jnp.cos(2 * omS * t + alp + bet) - 3 * jnp.cos(alp - bet)),
            jnp.sin(omS * t + alp)
            + Lrs
            / (4 * jnp.sqrt(3))
            * (jnp.sin(2 * omS * t + alp + bet) - 3 * jnp.sin(alp - bet)),
            -Lrs / 2 * jnp.cos(omS * t + bet),
        ]
    )


def sat2_pos(t):
    alp = ALPHATRIANGLE
    bet = BETATRIANGLE
    omS = YEAROMEGA
    Lrs = ARMLENGHTNORMALIZED
    return jnp.stack(
        [
            jnp.cos(omS * t + alp)
            + Lrs
            / (4 * jnp.sqrt(3))
            * (
                jnp.cos(2 * omS * t + alp + bet - 2 * jnp.pi / 3)
                - 3 * jnp.cos(alp - bet + 2 * jnp.pi / 3)
            ),
            jnp.sin(omS * t + alp)
            + Lrs
            / (4 * jnp.sqrt(3))
            * (
                jnp.sin(2 * omS * t + alp + bet - 2 * jnp.pi / 3)
                - 3 * jnp.sin(alp - bet + 2 * jnp.pi / 3)
            ),
            -Lrs / 2 * jnp.cos(omS * t + bet - 2 * jnp.pi / 3),
        ]
    )


def sat3_pos(t):
    alp = ALPHATRIANGLE
    bet = BETATRIANGLE
    omS = YEAROMEGA
    Lrs = ARMLENGHTNORMALIZED
    return jnp.stack(
        [
            jnp.cos(omS * t + alp)
            + Lrs
            / (4 * jnp.sqrt(3))
            * (
                jnp.cos(2 * omS * t + alp + bet + 2 * jnp.pi / 3)
                - 3 * jnp.cos(alp - bet - 2 * jnp.pi / 3)
            ),
            jnp.sin(omS * t + alp)
            + Lrs
            / (4 * jnp.sqrt(3))
            * (
                jnp.sin(2 * omS * t + alp + bet + 2 * jnp.pi / 3)
                - 3 * jnp.sin(alp - bet - 2 * jnp.pi / 3)
            ),
            -Lrs / 2 * jnp.cos(omS * t + bet + 2 * jnp.pi / 3),
        ]
    )


x_all = jnp.zeros((3, len(TIMES), 3))  # first index is xyz
x_all = x_all.at[0].set(sat1_pos(TIMES).T)
x_all = x_all.at[1].set(sat2_pos(TIMES).T)
x_all = x_all.at[2].set(sat3_pos(TIMES).T)

d_all = jnp.zeros((3, len(TIMES), 3, 3))  # first index is xyz
d_all = d_all.at[0].set(
    (x_all[0][:, None] - x_all[1][:, None])
    * (x_all[0][:, :, None] - x_all[1][:, :, None])
    - (x_all[0][:, None] - x_all[2][:, None])
    * (x_all[0][:, :, None] - x_all[2][:, :, None])
) / (2 * ARMLENGHTNORMALIZED**2)

d_all = d_all.at[1].set(
    (x_all[1][:, None] - x_all[2][:, None])
    * (x_all[1][:, :, None] - x_all[2][:, :, None])
    - (x_all[1][:, None] - x_all[0][:, None])
    * (x_all[1][:, :, None] - x_all[0][:, :, None])
) / (2 * ARMLENGHTNORMALIZED**2)

d_all = d_all.at[2].set(
    (x_all[2][:, None] - x_all[0][:, None])
    * (x_all[2][:, :, None] - x_all[0][:, :, None])
    - (x_all[2][:, None] - x_all[1][:, None])
    * (x_all[2][:, :, None] - x_all[1][:, :, None])
) / (2 * ARMLENGHTNORMALIZED**2)

d_A = (2 / jnp.sqrt(3)) * d_all[0]
d_E = -(2 / 3) * d_all[0] - (4 / 3) * d_all[1]


### Polarization tensors ###


def nhat(theta, phi):
    return jnp.stack(
        [jnp.sin(theta) * jnp.cos(phi), jnp.sin(theta) * jnp.sin(phi), jnp.cos(theta)]
    )


def get_theta_phi(vector):
    theta = jnp.arccos(vector[2])
    phi = jnp.arctan2(vector[1], vector[0])
    return theta, phi


def phat(theta, phi):
    return jnp.stack([jnp.sin(phi), -jnp.cos(phi), 0 * phi])


def qhat(theta, phi):
    return jnp.stack(
        [jnp.cos(theta) * jnp.cos(phi), jnp.cos(theta) * jnp.sin(phi), -jnp.sin(theta)]
    )


def tensor_ab(a, b):
    return jnp.einsum("ik, jk -> ijk", a, b)


def contractor_ab_a_b(ab, a, b):
    return jnp.einsum("ijk, ik, jk -> k", ab, a, b)


def scal_prod(a, b):
    return jnp.einsum("ij, ij -> j", a, b)


def e_plus(theta, phi):
    toret = tensor_ab(phat(theta, phi), phat(theta, phi)) - tensor_ab(
        qhat(theta, phi), qhat(theta, phi)
    )
    return toret / jnp.sqrt(
        2
    )  # This agrees with 2201.08782 and 2009.11845 and is different wrt Allen-Ottewill


def e_cross(theta, phi):
    toret = tensor_ab(phat(theta, phi), qhat(theta, phi)) + tensor_ab(
        qhat(theta, phi), phat(theta, phi)
    )
    return toret / jnp.sqrt(
        2
    )  # This agrees with 2201.08782 and 2009.11845 and is different wrt Allen-Ottewill


def NtildaE(f):
    L = ARMLENGHT
    Alisa = 3.0
    Plisa = 15.0

    fstar = 1 / (2 * jnp.pi * L / c)
    toret = (
        1
        / 2
        * (2 + jnp.cos(f / fstar))
        * (Plisa / L) ** 2
        * 10 ** (-24)
        * (1 + (0.002 / f) ** 4)
        + 2
        * (1 + jnp.cos(f / fstar) + (jnp.cos(f / fstar)) ** 2)
        * (Alisa / L) ** 2
        * 10 ** (-30)
        * (1 + (0.0004 / f) ** 2)
        * (1 + (f / 0.008) ** 4)
        * (1 / (2 * jnp.pi * f)) ** 4
    )
    return toret


def NtildaT(f):
    L = ARMLENGHT
    Alisa = 3.0
    Plisa = 15.0

    fstar = 1 / (2 * jnp.pi * L / c)
    toret = (1 - jnp.cos(f / fstar)) * (Plisa / L) ** 2 * 10 ** (-24) * (
        1 + (0.002 / f) ** 4
    ) + 2 * (1 - jnp.cos(f / fstar)) ** 2 * (Alisa / L) ** 2 * 10 ** (-30) * (
        1 + (0.0004 / f) ** 2
    ) * (
        1 + (f / 0.008) ** 4
    ) * (
        1 / (2 * jnp.pi * f)
    ) ** 4
    return toret


def PSD_E(f):
    return jnp.where(f != 0, NtildaE(f), 0.0)


def PSD_T(f):
    return jnp.where(f != 0, NtildaT(f), 0.0)


def noisegen(rng: Key, wPSD):
    T = TOBS
    N = NTIMESAMPS
    dt = DT
    freqs = rfftfreq(N, dt)
    S = wPSD(freqs)

    if N % 2 == 0:  # Nyquist bin if N even
        S = S.at[-1].set(0.0)

    rng_r, rng_i = jr.split(rng, 2)
    xf = (
        jnp.sqrt(S * N)[:, None]
        * (
            jr.normal(rng_r, shape=(N // 2 + 1, CHANNELS))
            + 1j * jr.normal(rng_i, shape=(N // 2 + 1, CHANNELS))
        )
        / jnp.sqrt(2.0)
    )
    return irfft(xf, axis=0)

In [ ]:
### Waveform ###
rngs = nnx.Rngs(42)
Parameters = Float[Array, "sources 6"]
Observation = Float[Array, "times 2"]

CHANNELS = 2
SOURCES = 1
AMPLITUDE_RANGE = (1e-25, 1e-24)
CIOTA_RANGE = (-1, 1)
F0_RANGE = (1e-4, 1e-2)
PHI0_RANGE = (0, 2 * np.pi)
THETA_RANGE = (0, np.pi)
PHI_RANGE = (0, 2 * np.pi)


def sample_joint(rng: Key) -> tuple[Parameters, Observation]:
    rng_A, rng_ciota, rng_f0, rng_phi0, rng_theta, rng_phi, rng_noise = jr.split(rng, 7)
    log_A = jr.uniform(
        rng_A,
        shape=(SOURCES,),
        minval=np.log(AMPLITUDE_RANGE[0]),
        maxval=np.log(AMPLITUDE_RANGE[1]),
    )
    ciota = jr.uniform(
        rng_ciota,
        shape=(SOURCES,),
        minval=CIOTA_RANGE[0],
        maxval=CIOTA_RANGE[1],
    )
    log_f0 = jr.uniform(
        rng_f0,
        shape=(SOURCES,),
        minval=np.log(F0_RANGE[0]),
        maxval=np.log(F0_RANGE[1]),
    )
    phi0 = jr.uniform(
        rng_phi0,
        shape=(SOURCES,),
        minval=PHI0_RANGE[0],
        maxval=PHI0_RANGE[1],
    )

    theta = jr.uniform(
        rng_theta,
        shape=(SOURCES,),
        minval=THETA_RANGE[0],
        maxval=THETA_RANGE[1],
    )

    phi = jr.uniform(
        rng_phi,
        shape=(SOURCES,),
        minval=PHI_RANGE[0],
        maxval=PHI_RANGE[1],
    )

    A = jnp.exp(log_A)
    f0 = jnp.exp(log_f0)
    x = jnp.stack([log_A, ciota, log_f0, phi0, theta, phi], axis=-1)

    Phi_GB = 2 * jnp.pi * f0 * TIMES[..., None] - phi0
    h_plus = A[None] * (1 + ciota[None] ** 2) * jnp.cos(Phi_GB)
    h_cross = 2 * A[None] * ciota[None] * jnp.sin(Phi_GB)

    F_plus_A = jnp.einsum("tij, ijs -> ts", d_A, e_plus(theta, phi))
    F_cross_A = jnp.einsum("tij, ijs -> ts", d_A, e_cross(theta, phi))

    F_plus_E = jnp.einsum("tij, ijs -> ts", d_E, e_plus(theta, phi))
    F_cross_E = jnp.einsum("tij, ijs -> ts", d_E, e_cross(theta, phi))

    h_A = (h_plus * F_plus_A + h_cross * F_cross_A).sum(-1)
    h_E = (h_plus * F_plus_E + h_cross * F_cross_E).sum(-1)

    h = jnp.stack([h_A, h_E], axis=-1)
    n = noisegen(rng_noise, PSD_E)
    y = h + n
    return x, y

@jax.jit
def log_posterior(x_flat: Parameters, y: Observation) -> Scalar:
    x = rearrange(x_flat, "... (sources p) -> ... sources p", p=6)
    log_A, ciota, log_f0, phi0, theta, phi = x[..., 0], x[..., 1], x[..., 2], x[..., 3], x[..., 4], x[..., 5]
    A, f0 = jnp.exp(log_A), jnp.exp(log_f0)

    Phi_GB = 2 * jnp.pi * f0 * TIMES[..., None] - phi0
    h_plus = A[None] * (1 + ciota[None] ** 2) * jnp.cos(Phi_GB)
    h_cross = 2 * A[None] * ciota[None] * jnp.sin(Phi_GB)

    F_plus_A = jnp.einsum("tij, ijs -> ts", d_A, e_plus(theta, phi))
    F_cross_A = jnp.einsum("tij, ijs -> ts", d_A, e_cross(theta, phi))

    F_plus_E = jnp.einsum("tij, ijs -> ts", d_E, e_plus(theta, phi))
    F_cross_E = jnp.einsum("tij, ijs -> ts", d_E, e_cross(theta, phi))

    h_A = (h_plus * F_plus_A + h_cross * F_cross_A).sum(-1)
    h_E = (h_plus * F_plus_E + h_cross * F_cross_E).sum(-1)

    h = jnp.stack([h_A, h_E], axis=-1)

    noisevars= PSD_E(jnp.abs(fftfreq(NTIMESAMPS,TOBS/NTIMESAMPS)))[1:,None]*NTIMESAMPS
    residual = jnp.abs(fft(y - h,axis=0)[1:])
    log_likelihood = -einsum(residual**2 / (2*noisevars), "... t c-> ...") 

    mask_a = (AMPLITUDE_RANGE[0] < A) * (A < AMPLITUDE_RANGE[1])
    mask_f0 = (F0_RANGE[0] < f0) * (f0 < F0_RANGE[1])
    mask_phi = (PHI_RANGE[0] < phi) * (phi < PHI_RANGE[1])
    mask_ciota = (CIOTA_RANGE[0] < ciota) * (ciota < CIOTA_RANGE[1])
    mask_theta = (THETA_RANGE[0] < theta) * (theta < THETA_RANGE[1])
    mask_phi0 = (PHI0_RANGE[0] < phi0) * (phi0 < PHI0_RANGE[1])
    log_prior = jnp.where(
        mask_a * mask_f0 * mask_phi * mask_ciota * mask_theta * mask_phi0,  -log_A - log_f0, -jnp.inf
    ).sum(-1)
    return log_prior + log_likelihood

In [ ]:
T = TOBS
N = NTIMESAMPS
dt = DT
freqs = rfftfreq(N, dt)

xtry,ytry = sample_joint(rngs.eval())
print(xtry)
print(jnp.exp(xtry[:,2]))
plt.loglog(freqs,jnp.abs(rfft(ytry, axis=0))[:,0], label="Data")
plt.show()

xtryflat=xtry.flatten()

wid=0
As=(1+np.linspace(-1,1,50)*1e-3)*float(xtryflat[wid])

logposts=[]

for i in range(len(As)):
    xflattemp=np.array(xtryflat)
    xflattemp[wid]=As[i]
    logposts.append(log_posterior(xflattemp,ytry))

plt.plot(As,jnp.array(logposts))
plt.axvline(x=xtryflat[wid],c='k')
plt.grid()
plt.show()

In [ ]:
# NUM_BLOCKS = 4
# NUM_HEADS = 8
# HIDDEN_DIM = 64 * NUM_HEADS
# PATCH_SIZE = 64
# LEARNING_RATE = 1e-4
# BATCH_SIZE = 512
# TOTAL_EXAMPLES = 1024 * 1000


# def adaptive_norm(
#     x: Float[Array, "... N D"],
#     scale: Float[Array, "... 1 D"],
#     shift: Float[Array, "... 1 D"],
# ):
#     x = x - x.mean(axis=-1, keepdims=True)
#     x = x / x.std(axis=-1, keepdims=True)
#     x = x * (1 + scale) + shift
#     return x


# class Modulation(nnx.Module):
#     def __init__(self, dim: int, *, rngs: nnx.Rngs):
#         self.linear = nnx.LinearGeneral(
#             in_features=dim,
#             out_features=(1, 3 * dim),
#             kernel_init=nnx.initializers.zeros,
#             bias_init=nnx.initializers.zeros,
#             rngs=rngs,
#         )

#     def __call__(self, y: Float[Array, "... D"]) -> tuple[Float[Array, "... 1 D"], ...]:
#         modulation = self.linear(nnx.silu(y))
#         shift, scale, gate = jnp.split(modulation, 3, axis=-1)
#         return shift, scale, gate


# class SinusoidalEmbed(nnx.Module):
#     def __init__(self, dim: int, period: float = 2 * np.pi, *, rngs: nnx.Rngs):
#         self.dim = dim
#         self.period = period
#         self.embed = FeedForward(2 * dim, dim, dim, rngs=rngs)

#     def __call__(self, t: Float[Array, "..."]) -> Float[Array, "... D"]:
#         freqs = jnp.exp(-jnp.log(self.period) * jnp.linspace(0, 1, self.dim))
#         angles = 2 * jnp.pi * freqs * t[..., None]
#         x = jnp.concat([jnp.sin(angles), jnp.cos(angles)], axis=-1)
#         x = self.embed(x)
#         return x


# class FeedForward(nnx.Sequential):
#     def __init__(
#         self,
#         input_dim: int,
#         hidden_dim: int,
#         output_dim: int,
#         activation=nnx.silu,
#         *,
#         rngs: nnx.Rngs,
#     ):
#         super().__init__(
#             nnx.Linear(input_dim, hidden_dim, rngs=rngs),
#             activation,
#             nnx.Linear(hidden_dim, output_dim, rngs=rngs),
#         )


# class CrossAttention(nnx.Module):
#     def __init__(self, dim: int, num_heads: int, use_bias=False, *, rngs: nnx.Rngs):
#         super().__init__()
#         assert dim % num_heads == 0, "dim should be divisible by num_heads"
#         self.num_heads = num_heads
#         self.qkv_proj_x = nnx.Linear(dim, dim * 3, use_bias=use_bias, rngs=rngs)
#         self.qkv_proj_c = nnx.Linear(dim, dim * 3, use_bias=use_bias, rngs=rngs)
#         self.out_proj_x = nnx.Linear(dim, dim, use_bias=use_bias, rngs=rngs)
#         self.out_proj_c = nnx.Linear(dim, dim, use_bias=use_bias, rngs=rngs)

#     def __call__(self, x: Float[Array, "... N D"], c: Float[Array, "... M D"]):
#         *_, N, Dx = x.shape
#         *_, M, Dc = c.shape
#         assert Dx == Dc, "x and c should have the same feature dimension"

#         x = self.qkv_proj_x(x)
#         c = self.qkv_proj_c(c)
#         h = jnp.concat([x, c], axis=-2)
#         qkv = rearrange(h, "... N (H D) -> ... N H D", H=self.num_heads)
#         q, k, v = jnp.split(qkv, 3, axis=-1)
#         h = nnx.dot_product_attention(q, k, v)
#         h = rearrange(h, "... N H D -> ... N (H D)")
#         x, c = jnp.split(h, [N], axis=-2)
#         x = self.out_proj_x(x)
#         c = self.out_proj_c(c)
#         return x, c


# class MMDiTBlock(nnx.Module):
#     def __init__(self, dim: int, num_heads: int, expand: int = 4, *, rngs: nnx.Rngs):
#         self.modulation1_x = Modulation(dim, rngs=rngs)
#         self.modulation1_c = Modulation(dim, rngs=rngs)
#         self.modulation2_x = Modulation(dim, rngs=rngs)
#         self.modulation2_c = Modulation(dim, rngs=rngs)
#         self.attention = CrossAttention(dim, num_heads, rngs=rngs)
#         self.mlp_x = FeedForward(dim, expand * dim, dim, rngs=rngs)
#         self.mlp_c = FeedForward(dim, expand * dim, dim, rngs=rngs)

#     def __call__(
#         self,
#         x: Float[Array, "... N D"],
#         c: Float[Array, "... M D"],
#         y: Float[Array, "... D"],
#     ):
#         # cross attention block
#         shift_x, scale_x, gate_x = self.modulation1_x(y)
#         shift_c, scale_c, gate_c = self.modulation1_c(y)
#         hx = adaptive_norm(x, scale_x, shift_x)
#         hc = adaptive_norm(c, scale_c, shift_c)
#         hx, hc = self.attention(hx, hc)
#         x = x + hx * gate_x
#         c = c + hc * gate_c

#         # feed forward blocks
#         shift_x, scale_x, gate_x = self.modulation2_x(y)
#         hx = adaptive_norm(x, scale_x, shift_x)
#         hx = self.mlp_x(hx)
#         x = x + hx * gate_x

#         shift_c, scale_c, gate_c = self.modulation2_c(y)
#         hc = adaptive_norm(c, scale_c, shift_c)
#         hc = self.mlp_c(hc)
#         c = c + hc * gate_c
#         return x, c


# class MMDiT(nnx.Module):
#     def __init__(
#         self,
#         x_dim: int,
#         c_dim: int,
#         hidden_dim: int,
#         num_heads: int,
#         num_blocks: int,
#         *,
#         rngs: nnx.Rngs,
#     ):
#         self.x_pos_embed = SinusoidalEmbed(hidden_dim, rngs=rngs)
#         self.x_embed = FeedForward(x_dim, hidden_dim, hidden_dim, rngs=rngs)

#         self.c_pos_embed = SinusoidalEmbed(hidden_dim, rngs=rngs)
#         self.c_embed = FeedForward(c_dim, hidden_dim, hidden_dim, rngs=rngs)

#         self.y_pos_embed = SinusoidalEmbed(hidden_dim, rngs=rngs)
#         self.y_embed = FeedForward(hidden_dim, hidden_dim, hidden_dim, rngs=rngs)

#         self.blocks = [
#             MMDiTBlock(hidden_dim, num_heads, rngs=rngs) for _ in range(num_blocks)
#         ]

#         self.out_modulation = Modulation(hidden_dim, rngs=rngs)
#         self.out_unembed = FeedForward(hidden_dim, hidden_dim, x_dim, rngs=rngs)

#     def __call__(
#         self,
#         x: Float[Array, "... N D"],
#         c: Float[Array, "... M D"],
#         t: Float[Array, "..."],
#     ) -> Float[Array, "... N D"]:
#         # embeddings
#         x_pos = jnp.linspace(0, 1, x.shape[-2])
#         x = self.x_embed(x) + self.x_pos_embed(x_pos)
#         c_pos = jnp.linspace(0, 1, c.shape[-2])
#         c = self.c_embed(c) + self.c_pos_embed(c_pos)
#         y = self.y_embed(self.y_pos_embed(t))

#         # cross attention blocks
#         for block in self.blocks:
#             x, c = block(x, c, y)

#         # unembedging
#         shift, scale, gate = self.out_modulation(y)
#         x = adaptive_norm(x, scale, shift)
#         x = self.out_unembed(x)
#         return x


# class Flow(nnx.Module):
#     def __init__(self, *, rngs: nnx.Rngs):
#         self.DiT = MMDiT(
#             x_dim=6,
#             c_dim=2 * PATCH_SIZE,
#             hidden_dim=HIDDEN_DIM,
#             num_heads=NUM_HEADS,
#             num_blocks=NUM_BLOCKS,
#             rngs=rngs,
#         )

#     @nnx.jit
#     def __call__(self, x: Parameters, t: Scalar, y: Observation) -> Parameters:
#         # c = rearrange(y, "... (T P) C -> ... T (P C)", P=PATCH_SIZE)
#         _, _, c = jsp.signal.stft(y, nperseg=PATCH_SIZE - 1, axis=-2)
#         c = rearrange(c, "... F C T -> ... T (C F)")
#         c = jnp.concatenate([c.real, c.imag], axis=-1)
#         return self.DiT(x, c, t)

#     @nnx.jit
#     def ode_step(
#         self, x: Parameters, t: Scalar, y: Observation, dt: float
#     ) -> Parameters:
#         k1 = self(x, t, y)
#         k2 = self(x + k1 * dt / 2, t + dt / 2, y)
#         k3 = self(x + k2 * dt / 2, t + dt / 2, y)
#         k4 = self(x + k3 * dt, t + dt, y)
#         x = x + (k1 + 2 * k2 + 2 * k3 + k4) * dt / 6
#         return x


# @jax.jit
# def get_train_batch(rng: Key) -> tuple[Parameters, Scalar, Observation, Parameters]:
#     def phi(t: Scalar, x1: Parameters, x0: Parameters) -> Parameters:
#         return x1 * t + x0 * (1 - t)

#     def train_sample(rng: Key) -> tuple[Parameters, Scalar, Observation, Parameters]:
#         rng_xy, rng_x0, rng_t = jr.split(rng, 3)
#         x1, y = sample_joint(rng_xy)
#         x0 = jr.normal(rng_x0, x1.shape)
#         t = jr.uniform(rng_t, minval=0.0, maxval=1.0)

#         xt = x1 * t + x0 * (1 - t)
#         dx = jax.jacobian(phi)(t, x1, x0)
#         return xt, t, y, dx

#     return jax.vmap(train_sample)(jr.split(rng, BATCH_SIZE))


# @nnx.jit
# def train_step(
#     model: Flow,
#     optimizer: nnx.Optimizer,
#     batch: tuple[Parameters, Scalar, Observation, Parameters],
# ) -> Scalar:
#     def loss_fn(model):
#         xt, t, y, dx = batch
#         return jnp.mean((model(xt, t, y) - dx) ** 2)

#     loss, grads = nnx.value_and_grad(loss_fn)(model)
#     optimizer.update(grads)
#     return loss


# flow = Flow(rngs=rngs)
# optimizer = nnx.Optimizer(flow, optax.adamw(learning_rate=LEARNING_RATE))

# for step in (pbar := tqdm(range(TOTAL_EXAMPLES // BATCH_SIZE))):
#     batch = get_train_batch(rngs.batch())
#     loss = train_step(flow, optimizer, batch)
#     pbar.set_postfix(loss=loss.item())

In [ ]:
RUNS = 10
SAMPLES = 1024 
DIFFUSIONSTEPS = 16
MCMCWALKERS = 64
MCMCDISCARD = 1000
MCMCTHIN = 10


# def sample_from_flow(y: Observation):
#     t = jnp.zeros((SAMPLES,))
#     x = jr.normal(rngs.eval(), (SAMPLES, SOURCES, 6))
#     y = jnp.broadcast_to(y, (SAMPLES, *y.shape))
#     dt = 1.0 / DIFFUSIONSTEPS
#     for _ in tqdm(range(DIFFUSIONSTEPS)):
#         x = flow.ode_step(x, t, y, dt)
#         t += dt
#     x = rearrange(x, "... S P -> ... (S P)")
#     return np.array(x)


def sample_from_mcmc(y: Observation, x_true_flat: Parameters):
    p0 = np.random.randn(MCMCWALKERS, x_true_flat.shape[-1])
    sampler = EnsembleSampler(MCMCWALKERS, x_true_flat.shape[-1], log_posterior, args=(y,))
    sampler.run_mcmc(p0, nsteps=MCMCTHIN * SAMPLES // MCMCWALKERS + MCMCDISCARD, progress=True)
    x = sampler.get_chain(flat=True, discard=MCMCDISCARD, thin=10)
    return x


def mirror(x_flat):
    x = rearrange(x_flat, "... (S P) -> ... S P", P=3)
    x_mirrored = x[..., ::-1, :]
    x_mirrored_flat = rearrange(x_mirrored, "... S P -> ... (S P)")
    return x_mirrored_flat


for run in range(RUNS):
    x_true, y = sample_joint(rngs.eval())
    x_true_flat = rearrange(x_true, "... N P -> ... (N P)")

    # print("Running flow sampling...")
    # generated_samples = sample_from_flow(y)
    print("Running MCMC...")
    mcmc_samples = sample_from_mcmc(y, x_true_flat)
    print()

    param_names = sum(([f"A_{i}", f"ciota_{i}", f"$f0_{i}$", f"$phi0_{i}$", f"$theta_{i}$", f"$phi_{i}$"] for i in range(SOURCES)), [])
    fig = corner(mcmc_samples, labels=param_names, truths=x_true_flat, color="red")
    # fig = corner(generated_samples, labels=param_names, truths=x_true_flat, color="red")
    # fig = corner(mirror(generated_samples), color="orange", fig=fig)
    # fig = corner(mcmc_samples, color="blue", fig=fig)
    plt.show()